# Trabajo 3 Sistemas de Big Data
## Análisis de datos con Big Data

El objetivo de este trabajo es que los profesores-alumnos se familiaricen con con uso de las distintas técnicas de análisis de datos en Spark (las cuales son aplicadas por gran parte de la industria en el contexto del Big Data). Para ello, se presenta un dataset de tamaño reducido (para que pueda ejecutarse en google colab) y se proponen una serie de pasos. El alumno debe completar el código correspondiente para terminar el cuaderno y responder a las preguntas formuladas (en negrita) utilizando celdas de texto.

La estructura de este cuaderno es la siguiente:
1.   Instalación y puesta en marcha de un cluster de Spark.
2.   Carga y procesamiento del dataset
3.   Determinación del algoritmo que proporciona el mejor resultado para este problema (el alumno debe probar todos los algoritmos que considere para obtener una buena solución).
4.   Discusión de los resultados.



In [1]:
## Solicitar acceso y montar en el sistema tu directorio de Google Drive
# Esto permitirá la persistencia de datos entre distintas sesiones
import os, sys
from google.colab import drive
drive.mount('/content/drive')

# Creamos un directorio para este notebook y lo asociamos
drive_path = '/content/drive/MyDrive/Colab Notebooks/ia-bd-m4-sistemas-de-big-data'
nb_path = '/content/ia-bd-m4-sistemas-de-big-data'

if not os.path.exists(drive_path):
    os.makedirs(drive_path)
os.symlink(drive_path, nb_path)
sys.path.insert(0,nb_path)

%cd $nb_path

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/ia-bd-m4-sistemas-de-big-data


## Instalación de spark en Google Colab

In [2]:
# Spark está escrito en el lenguaje de programación Scala, por lo que requiere 
# de una máquina virtual de Java (JVM) para funcionar. Por lo tanto, lo primero
# es instalar java:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [3]:
## Instalación de Apache Spark en Google Colab (Ejecutar solo si no se ha realizado nunca)
# Nota: Puede tardar unos minutos (se paciente)
# El siguiente paso es elegir una versión reciente de spark
# En este notebook, se usará spark versión 3.1.2, la cual puede descargarse en:
spark_file = 'spark-3.1.2-bin-hadoop3.2.tgz'
spark_url = 'https://archive.apache.org/dist/spark/spark-3.1.2/' + spark_file

# A continuación, descargamos la versión elegida de spark:
import os # Libreria de manejo del sistema operativo
os.system("wget -q {spark_url} -P " + nb_path) # Realizamos la descarga
os.system("tar xf " + nb_path + "/" + spark_file) # Descomprimimos el fichero .tgz

# Realizamos la instalación de pyspark utilizando la herramienta pip
!pip install --target=$nb_path -q pyspark
!pip install -q findspark

     |████████████████████████████████| 281.3 MB 45 kB/s 
     |████████████████████████████████| 199 kB 58.5 MB/s 


In [4]:
# Damos permisos de ejecución
!chmod -R +x ./pyspark/

# Finalmente, es necesario definir algunas variables de entorno en el sistema 
# operativo para poder usar spark correctamente:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = nb_path + "/pyspark"

## Cargar el conjunto de datos que se utilizará

En este caso utilizaremos un conjunto de datos completo con un gran número de indicadores económicos para predecir el si se debe invertir en bolsa (*SPY17VAR.csv*). El conjunto de datos contiene la fecha, el precio de apertura, el precio de cierre y el máximo y el mínimo durante la sesión. También contiene numerosos indicadores financieros, la fecha desagregada en su información y la clase a predecir (la cual indica si es el momento de invertir o no).



In [5]:
# Descargar el dataset que se utilizará utilizando un enlace compartido de google drive
# El fichero puede descargarse directamente de la fuente proporcionada, o se
# puede utilizar el siguiente código para descargarlo de google drive:
# URL: https://drive.google.com/file/d/10hUVsbe9rj9dInaZl9UEikvaOkSiA-5L
!gdown --id 10hUVsbe9rj9dInaZl9UEikvaOkSiA-5L

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=10hUVsbe9rj9dInaZl9UEikvaOkSiA-5L
To: /content/drive/MyDrive/Colab Notebooks/ia-bd-m4-sistemas-de-big-data/SPY17VAR.csv
100% 420k/420k [00:00<00:00, 52.4MB/s]


**Pregunta: En base a los datos que hemos cargado, ¿estamos ante un problema de clustering, clasificación o regresión?**

*Solución:* Desde mi punto de vista, estamos ante un problema de clasificación ya que se deber predecir el valor de una variable (momento de invertir) a partir de un conjunto de variables (datos suministrados)



Creamos un cluster de Spark e importamos el dataframe actual en Spark.

In [6]:
# Importar pyspark.sql
from pyspark.sql import*

# Importar SparkContext and SparkConf
from pyspark import SparkContext, SparkConf

In [7]:
# Establecer las propiedades de Spark: 
# - URL de conexión
# - Nombre de la aplicación
conf = SparkConf().setMaster("local").setAppName("SBD-Trabajo3")

# Iniciar un cluster de Spark (puede tardar unos minutos)
# Comprobar si ya existe este cluster y en el caso contrario crear uno nuevo 
sc = SparkContext.getOrCreate(conf=conf)

# Mostramos el cluster creado
sc

<SparkContext master=local appName=SBD-Trabajo3>

In [8]:
# Inicializar SQLContext a partir del cluster Spark creado anteriormente
sqlContext = SQLContext(sc)

# Creamos un dataframe a partir del archivo CSV descargado anteriormente y 
# que contiene el dataset que utilizaremos en esta sesión
df = sqlContext.read.csv('SPY17VAR.csv', header=True, sep=",", inferSchema = "true")

# Mostrar el contenido del dataframe (las 5 primeras observaciones)
df.show(5)

/content/ia-bd-m4-sistemas-de-big-data/pyspark/sql/context.py:114: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


+-------+------+------+------+------+-----+----+----+------+------+----+-----+----+----+-----+---+----+-----+-------+-----+-------+-----------+-------+---------+----------+-----------------+----------------+
|   DATE|  OPEN|   MAX|   MIN| CLOSE|CLASS|   2|  42|    45|    48|  68|   75|  88| 139|  171|172| 179|  187|    218|  221|    223|        231|    237|date.year|date.month|date.day-of-month|date.day-of-week|
+-------+------+------+------+------+-----+----+----+------+------+----+-----+----+----+-----+---+----+-----+-------+-----+-------+-----------+-------+---------+----------+-----------------+----------------+
|3/26/07| 143.5|143.65|142.09| 143.2|    1|2.87|0.19|1.6351|1.3908|0.97|14.54|1.72|0.95|-0.08|  I| 1.4|14.64|91.8447|23.43| 3.3864|45418933114| 4.7209|        7|         3|               26|               1|
|3/27/07|143.12|143.16|142.39|142.86|    1|2.84|0.19|1.5526|1.3792|0.95|15.58|5.17| 0.9|-0.07|  I|0.98|14.64| 90.701|23.49|  0.762|44525544754| 1.6001|        7|       

## Pre-procesamiento del dataset y obtención de conjuntos de entrenamiento y test

En este caso, para simplicidad del problema, sabemos que solo tres de las variables del dataset proporcionan información relevante para este problema ['45', '75', '171']. Vamos, por tanto, a reducir el conjunto de datos a solo estas tres variables más la clase a predecir ('CLASS').


In [9]:
df = df.select('45', '75', '171', 'CLASS')
df.show(5)

+------+-----+-----+-----+
|    45|   75|  171|CLASS|
+------+-----+-----+-----+
|1.6351|14.54|-0.08|    1|
|1.5526|15.58|-0.07|    1|
|1.5573|14.55|-0.07|    1|
|1.5436| 14.4|-0.06|    1|
|1.6172|14.33|-0.06|    1|
+------+-----+-----+-----+
only showing top 5 rows



A continuación, utilizaremos el *VectorAssembler* para generar una nueva columna en el dataframe la cual tendrá un vector del tipo DenseVector conteniendo todas las características del dataset. Recordad que este era un paso necesario antes de aplicar cualquier algoritmo de ML de la biblioteca [MLlib de Spark](https://spark.apache.org/docs/latest/ml-guide.html).

In [10]:
# Completa con el código correspondiente
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
  inputCols = ["45", "75", "171", "CLASS"], 
  outputCol = "features"
)
df = assembler.transform(df)

# Mostramos la nueva columa "features" para las 10 primeras filas del dataset
df.select("features").show(10, truncate=False)



+------------------------+
|features                |
+------------------------+
|[1.6351,14.54,-0.08,1.0]|
|[1.5526,15.58,-0.07,1.0]|
|[1.5573,14.55,-0.07,1.0]|
|[1.5436,14.4,-0.06,1.0] |
|[1.6172,14.33,-0.06,1.0]|
|[1.5535,14.42,-0.06,1.0]|
|[1.5802,14.32,-0.05,1.0]|
|[1.5011,14.62,-0.05,1.0]|
|[1.434,14.65,-0.05,1.0] |
|[1.3556,14.57,-0.05,1.0]|
+------------------------+
only showing top 10 rows



In [11]:
# Mostramos la nueva columa "features" para las 5 primeras filas del dataset
df.select("features").show(5, truncate=False)

+------------------------+
|features                |
+------------------------+
|[1.6351,14.54,-0.08,1.0]|
|[1.5526,15.58,-0.07,1.0]|
|[1.5573,14.55,-0.07,1.0]|
|[1.5436,14.4,-0.06,1.0] |
|[1.6172,14.33,-0.06,1.0]|
+------------------------+
only showing top 5 rows



Posteriormente procedemos a estandarizar las entradas numéricas con las que contamos. Este es un proceso muy recomendable para la aplicación de algunos de los algoritmos de clasificación, como por ejemplo para la regresión logística. El escalado que aplicaremos será una estandarízación de desviación estándar la unidad.

In [12]:
# Completa con el código correspondiente
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features",outputCol='standardized')
fit_scaler = scaler.fit(df)
df = fit_scaler.transform(df)

# Mostramos la nueva columa "standardized" para las 5 primeras filas del dataset
df.select("standardized").show(5, truncate=False)



+-----------------------------------------------------------------------------+
|standardized                                                                 |
+-----------------------------------------------------------------------------+
|[1.7531128452981628,2.413142779833078,-0.1250086814089131,2.363105222288687] |
|[1.6646584328847946,2.585747215254426,-0.10938259623279899,2.363105222288687]|
|[1.6696976539556168,2.414802437865976,-0.10938259623279899,2.363105222288687]|
|[1.655008860621518,2.389907567372512,-0.09375651105668484,2.363105222288687] |
|[1.7339209182412016,2.378289961142229,-0.09375651105668484,2.363105222288687]|
+-----------------------------------------------------------------------------+
only showing top 5 rows



Ahora vamos a dividir el dataset en un conjunto de entrenamiento y otro de test. Gracias a esto, podremos estimar cada modelo sobre el conjunto de entrenamiento, utilizando el conjunto de test para validar los resultados obtenidos.

In [13]:
# Completa con el código correspondiente

# Dividimos entre entrenamiento (70%) y test (30%)
(train_df, test_df) = df.randomSplit([0.7, 0.3], seed=100)

print("Número de filas df entrenamiento: {train:d}".format(train=train_df.count()))
print("Número de filas df test: {test:d}".format(test=test_df.count()))



Número de filas df entrenamiento: 1943
Número de filas df test: 840


## Determinación del mejor algoritmo de clasificación

A continuación prueba a entrenar aquellos algoritmos de clasificación que consideres adecuados sobre el conjunto de entrenamiento y comprueba su rendimiento sobre el conjunto de test. Recomendamos que el alumno pruebe a ejecutar todos los algoritmos de clasificación estudiados, así como que realice un ajuste de los parámetros que considere adecuados de estos para mejorar el resultado obtenido. Recordamos que estos algoritmos se estudiaron en el cuaderno "3-Algoritmos de clasificación.ipynb" de la semana 10: Inteligencia artificial en el análisis de datos.

Finalmente, en la sección de resultados, deberá explicar que elección realizar en base a los resultados obtenidos.

**¡Cuidado, algunos algoritmos obtienen resultados muy malos en este dataset!**



In [ ]:
# Completa con el código correspondiente

# Añade tantas celdas de código y de texto (con las explicaciones) como consideres oportundas

#Regresión Logística
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(labelCol="CLASS", featuresCol="standardized", 
                        maxIter=100)

# Definimos los parámetros del grid donde se buscarán los parámetros óptimos
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [1, 0.1, 0.01, 0.001]) \
    .build()
  
evaluator = BinaryClassificationEvaluator(labelCol="CLASS", rawPredictionCol="prediction")

# Definimos la cross-validación
crossval = CrossValidator(estimator=lr,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Ejecutar la cross-validación y elegir el mejor conjunto de parámetros
cvModel = crossval.fit(train_df)

print("RegParam: " + str(cvModel.bestModel.getRegParam()))




RegParam: 0.1


In [ ]:
# Entrenar un modelo de regresión logistica binario (2 clases)
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="CLASS", featuresCol="standardized", 
                        maxIter=100, regParam=0.001)

lr_model = lr.fit(train_df)
pred = lr_model.transform(test_df)

# Mostramos las clases reales junto con las predicciones realizadas
pred.select("CLASS", "features", "prediction").show(10)

+-----+--------------------+----------+
|CLASS|            features|prediction|
+-----+--------------------+----------+
|    1|[0.87,15.48,-0.36...|       1.0|
|    1|[0.9604,24.95,-0....|       1.0|
|    0|[0.9703,25.9,-0.5...|       0.0|
|    1|[0.9749,15.23,-0....|       1.0|
|    1|[0.976,25.0,-0.13...|       1.0|
|    1|[0.9968,24.85,-0....|       1.0|
|    1|[1.0001,25.05,-0....|       1.0|
|    1|[1.0197,25.1,-0.1...|       1.0|
|    1|[1.0264,15.43,-0....|       1.0|
|    1|[1.0402,24.85,-0....|       1.0|
+-----+--------------------+----------+
only showing top 10 rows



In [ ]:
# Mostrar los coeficientes y los términos de interceptación de la regresión logistica
print("Coeficientes (1 por variable): " + str(lr_model.coefficientMatrix))
print("Terminos de interceptación: " + str(lr_model.interceptVector))

Coeficientes (1 por variable): DenseMatrix([[ 0.04655802,  0.12780322, -0.12152324,  4.54149804]])

Terminos de interceptación: [-5.320352922829045]


In [ ]:
#EVALUACIÓN DEL MODELO
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="CLASS", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("Accuracy: {}".format(accuracy))

Accuracy: 1.0


In [ ]:
#mostramos la matriz de confusión
from pyspark.mllib.evaluation import MulticlassMetrics

# Selecionamos solo las prediciones y los valores reales (CLASS)
preds_and_labels = pred.select(['prediction','CLASS'])
# Hacemos un casting de entero a float para la columna CLASS
preds_and_labels = preds_and_labels.withColumn("CLASS", preds_and_labels["CLASS"].cast('float'))
# Pasamos el resultado anterior a un objeto RDD (Resilient Distributed Datasets) de tuplas
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

# Mostramos la matriz de confusión
print(metrics.confusionMatrix().toArray())

/content/ia-bd-m4-sistemas-de-big-data/pyspark/sql/context.py:159: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


[[187.   0.]
 [  0. 653.]]


In [ ]:
#Máquinas de Vectores de Soporte (SVM) - Lineales
from pyspark.ml.classification import LinearSVC
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lsvc = LinearSVC(labelCol="CLASS", featuresCol="standardized", 
                 maxIter=100)

# Definimos los parámetros del grid donde se buscarán los parámetros óptimos
paramGrid = ParamGridBuilder() \
    .addGrid(lsvc.regParam, [1, 0.1, 0.01, 0.001]) \
    .build()
  
evaluator = BinaryClassificationEvaluator(labelCol="CLASS", rawPredictionCol="prediction")

# Definimos la cross-validación
crossval = CrossValidator(estimator=lsvc,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Ejecutar la cross-validación y elegir el mejor conjunto de parámetros
cvModel = crossval.fit(train_df)

print("RegParam: " + str(cvModel.bestModel.getRegParam()))


#ENTRENAMOS EL MODELO
from pyspark.ml.classification import LinearSVC

lsvc = LinearSVC(labelCol="CLASS", featuresCol="standardized", 
                 maxIter=100, regParam=0.001)

lsvc_model = lsvc.fit(train_df)
pred = lsvc_model.transform(test_df)

# Mostramos las clases reales junto con las predicciones realizadas
print("_____________________________________________")
print("Mostramos las clases reales junto con las predicciones realizadas")
pred.select("CLASS", "features", "prediction").show(10)

#EVALUAMOS EL MODELO

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="CLASS", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("_____________________________________________")
print("evaluamos el modelo")
print("Accuracy: {}".format(accuracy))


#OBTENEMOS LA MATRIZ DE CONFUSIÓN
from pyspark.mllib.evaluation import MulticlassMetrics

# Selecionamos solo las prediciones y los valores reales (forgery)
preds_and_labels = pred.select(['prediction','CLASS'])
# Hacemos un casting de entero a float para la columna forgery
preds_and_labels = preds_and_labels.withColumn("CLASS", preds_and_labels["CLASS"].cast('float'))
# Pasamos el resultado anterior a un objeto RDD (Resilient Distributed Datasets) de tuplas
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

# Mostramos la matriz de confusión
print("_____________________________________________")
print("matriz de confusión")
print(metrics.confusionMatrix().toArray())

RegParam: 1.0
_____________________________________________
Mostramos las clases reales junto con las predicciones realizadas
+-----+--------------------+----------+
|CLASS|            features|prediction|
+-----+--------------------+----------+
|    1|[0.87,15.48,-0.36...|       1.0|
|    1|[0.9604,24.95,-0....|       1.0|
|    0|[0.9703,25.9,-0.5...|       0.0|
|    1|[0.9749,15.23,-0....|       1.0|
|    1|[0.976,25.0,-0.13...|       1.0|
|    1|[0.9968,24.85,-0....|       1.0|
|    1|[1.0001,25.05,-0....|       1.0|
|    1|[1.0197,25.1,-0.1...|       1.0|
|    1|[1.0264,15.43,-0....|       1.0|
|    1|[1.0402,24.85,-0....|       1.0|
+-----+--------------------+----------+
only showing top 10 rows

_____________________________________________
evaluamos el modelo
Accuracy: 1.0


/content/ia-bd-m4-sistemas-de-big-data/pyspark/sql/context.py:159: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


_____________________________________________
matriz de confusión
[[187.   0.]
 [  0. 653.]]


In [14]:
#ARBOLES DE DESICIÓN - CLASIFICACIÓN RANDOM FOREST
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

rf = RandomForestClassifier(labelCol="CLASS", featuresCol="standardized")

# Definimos los parámetros del grid donde se buscarán los parámetros óptimos
paramGrid = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [3, 6, 10]) \
    .addGrid(rf.numTrees, [50, 100, 150, 250]) \
    .build()

evaluator = BinaryClassificationEvaluator(labelCol="CLASS", rawPredictionCol="prediction")

# Definimos la cross-validación
crossval = CrossValidator(estimator=rf,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Ejecutar la cross-validación y elegir el mejor conjunto de parámetros
cvModel = crossval.fit(train_df)
print("_____________________________________________")
print("_____________________________________________")
print("matriz de confusión")
print("MaxDepth: " + str(cvModel.bestModel.getMaxDepth()))
print("NumTrees: " + str(cvModel.bestModel.getNumTrees))

# Entrenar un clasificador Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="CLASS", featuresCol="standardized", 
                            maxDepth=10, numTrees=100, seed=10)

rf_model = rf.fit(train_df)
pred = rf_model.transform(test_df)

# Mostramos las clases reales junto con las predicciones realizadas
print("_____________________________________________")
print("_____________________________________________")
print("Mostramos las clases reales junto con las predicciones realizadas")
pred.select("CLASS", "features", "prediction").show(10)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="CLASS", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("_____________________________________________")
print("_____________________________________________")
print("Evaluación del modelo")
print("Accuracy: {}".format(accuracy))

from pyspark.mllib.evaluation import MulticlassMetrics

# Selecionamos solo las prediciones y los valores reales (forgery)
preds_and_labels = pred.select(['prediction','CLASS'])
# Hacemos un casting de entero a float para la columna forgery
preds_and_labels = preds_and_labels.withColumn("CLASS", preds_and_labels["CLASS"].cast('float'))
# Pasamos el resultado anterior a un objeto RDD (Resilient Distributed Datasets) de tuplas
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

# Mostramos la matriz de confusión
print("_____________________________________________")
print("_____________________________________________")
print("Mostramos la matriz de confusión")
print(metrics.confusionMatrix().toArray())

_____________________________________________
_____________________________________________
matriz de confusión
MaxDepth: 3
NumTrees: 50
_____________________________________________
_____________________________________________
Mostramos las clases reales junto con las predicciones realizadas
+-----+--------------------+----------+
|CLASS|            features|prediction|
+-----+--------------------+----------+
|    1|[0.87,15.48,-0.36...|       1.0|
|    1|[0.9604,24.95,-0....|       1.0|
|    0|[0.9703,25.9,-0.5...|       0.0|
|    1|[0.9749,15.23,-0....|       1.0|
|    1|[0.976,25.0,-0.13...|       1.0|
|    1|[0.9968,24.85,-0....|       1.0|
|    1|[1.0001,25.05,-0....|       1.0|
|    1|[1.0197,25.1,-0.1...|       1.0|
|    1|[1.0264,15.43,-0....|       1.0|
|    1|[1.0402,24.85,-0....|       1.0|
+-----+--------------------+----------+
only showing top 10 rows

_____________________________________________
_____________________________________________
Evaluación del modelo
Accur

/content/ia-bd-m4-sistemas-de-big-data/pyspark/sql/context.py:159: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


_____________________________________________
_____________________________________________
Mostramos la matriz de confusión
[[187.   0.]
 [  0. 653.]]


In [15]:
# Árboles de Decisión - Gradient-boosted tree classifier

from pyspark.ml.classification import GBTClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

gbt = GBTClassifier(labelCol="CLASS", featuresCol="standardized")

# Definimos los parámetros del grid donde se buscarán los parámetros óptimos
paramGrid = ParamGridBuilder() \
    .addGrid(gbt.maxDepth, [3, 6, 10]) \
    .build()

evaluator = BinaryClassificationEvaluator(labelCol="CLASS", rawPredictionCol="prediction")

# Definimos la cross-validación
crossval = CrossValidator(estimator=gbt,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Ejecutar la cross-validación y elegir el mejor conjunto de parámetros
cvModel = crossval.fit(train_df)

print("_____________________________________________")
print("_____________________________________________")
print("Árboles de Decisión - Gradient-boosted tree classifier")
print("MaxDepth: " + str(cvModel.bestModel.getMaxDepth()))

# Entrenar un clasificador Random Forest
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(labelCol="CLASS", featuresCol="standardized", 
                            maxDepth=10, seed=10)

gbt_model = gbt.fit(train_df)
pred = gbt_model.transform(test_df)

# Mostramos las clases reales junto con las predicciones realizadas
print("_____________________________________________")
print("_____________________________________________")
print("Mostramos las clases reales junto con las predicciones realizadas")
pred.select("CLASS", "features", "prediction").show(10)

#EVALUACIÓN DEL MODELO
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="CLASS", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("_____________________________________________")
print("_____________________________________________")
print("EVALUACIÓN DEL MODELO")
print("Accuracy: {}".format(accuracy))

from pyspark.mllib.evaluation import MulticlassMetrics

# Selecionamos solo las prediciones y los valores reales (forgery)
preds_and_labels = pred.select(['prediction','CLASS'])
# Hacemos un casting de entero a float para la columna forgery
preds_and_labels = preds_and_labels.withColumn("CLASS", preds_and_labels["CLASS"].cast('float'))
# Pasamos el resultado anterior a un objeto RDD (Resilient Distributed Datasets) de tuplas
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

# Mostramos la matriz de confusión
print("_____________________________________________")
print("_____________________________________________")
print("Mostramos la matriz de confusión")
print(metrics.confusionMatrix().toArray())

_____________________________________________
_____________________________________________
Árboles de Decisión - Gradient-boosted tree classifier
MaxDepth: 3
_____________________________________________
_____________________________________________
Mostramos las clases reales junto con las predicciones realizadas
+-----+--------------------+----------+
|CLASS|            features|prediction|
+-----+--------------------+----------+
|    1|[0.87,15.48,-0.36...|       1.0|
|    1|[0.9604,24.95,-0....|       1.0|
|    0|[0.9703,25.9,-0.5...|       0.0|
|    1|[0.9749,15.23,-0....|       1.0|
|    1|[0.976,25.0,-0.13...|       1.0|
|    1|[0.9968,24.85,-0....|       1.0|
|    1|[1.0001,25.05,-0....|       1.0|
|    1|[1.0197,25.1,-0.1...|       1.0|
|    1|[1.0264,15.43,-0....|       1.0|
|    1|[1.0402,24.85,-0....|       1.0|
|    1|[1.0488,23.4,-0.1...|       1.0|
|    1|[1.0567,24.8,-0.0...|       1.0|
|    1|[1.058,18.5,-0.51...|       1.0|
|    1|[1.0607,24.4,-0.1...|       1.0|
|  

/content/ia-bd-m4-sistemas-de-big-data/pyspark/sql/context.py:159: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning,


_____________________________________________
_____________________________________________
Mostramos la matriz de confusión
[[187.   0.]
 [  0. 653.]]


In [ ]:
#REDES NEURONALES
# Entrenar un Multilayer Perceptron (un tipo de red neuronal feedforward)
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Especificamos las capas de neuronas de la red neuronal
# Capa de entrada de tamaño 4 (las features), dos capas intermedias de tamaños
# 5 y 4 respectivamente, y finalmente una capa de salida de 2 (las clases)
layers = [4, 10, 10, 2]

mlp = MultilayerPerceptronClassifier(labelCol="CLASS", featuresCol="standardized", 
                                     maxIter=100, layers=layers, seed=10)

mlp_model = mlp.fit(train_df)
pred = mlp_model.transform(test_df)

# Mostramos las clases reales junto con las predicciones realizadas
print("_____________________________________________")
print("_____________________________________________")
print("Mostramos las clases reales junto con las predicciones realizadas")
pred.select("CLASS", "features", "prediction").show(10)

#EVALUACIÓN DEL MODELO
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="CLASS", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(pred)
print("_____________________________________________")
print("_____________________________________________")
print("EVALUACIÓN DEL MODELO")
print("Accuracy: {}".format(accuracy))

#OBTENEMOS LA MATRIZ DE CONFUSIÓN
from pyspark.mllib.evaluation import MulticlassMetrics

# Selecionamos solo las prediciones y los valores reales (forgery)
preds_and_labels = pred.select(['prediction','CLASS'])
# Hacemos un casting de entero a float para la columna forgery
preds_and_labels = preds_and_labels.withColumn("CLASS", preds_and_labels["CLASS"].cast('float'))
# Pasamos el resultado anterior a un objeto RDD (Resilient Distributed Datasets) de tuplas
metrics = MulticlassMetrics(preds_and_labels.rdd.map(tuple))

# Mostramos la matriz de confusión
print("_____________________________________________")
print("_____________________________________________")
print("MOSTRAMOS LA MATRIZ DE CONFUSIÓN")
print(metrics.confusionMatrix().toArray())


_____________________________________________
_____________________________________________
Mostramos las clases reales junto con las predicciones realizadas
+-----+--------------------+----------+
|CLASS|            features|prediction|
+-----+--------------------+----------+
|    1|[0.87,15.48,-0.36...|       1.0|
|    1|[0.9604,24.95,-0....|       1.0|
|    0|[0.9703,25.9,-0.5...|       0.0|
|    1|[0.9749,15.23,-0....|       1.0|
|    1|[0.976,25.0,-0.13...|       1.0|
|    1|[0.9968,24.85,-0....|       1.0|
|    1|[1.0001,25.05,-0....|       1.0|
|    1|[1.0197,25.1,-0.1...|       1.0|
|    1|[1.0264,15.43,-0....|       1.0|
|    1|[1.0402,24.85,-0....|       1.0|
+-----+--------------------+----------+
only showing top 10 rows

_____________________________________________
_____________________________________________
EVALUACIÓN DEL MODELO
Accuracy: 1.0
_____________________________________________
_____________________________________________
MOSTRAMOS LA MATRIZ DE CONFUSIÓN
[[1

## Resultados

**Finalmente, hablaremos sobre los resultados obtenidos. Agrega tantas celdas de texto como consideres a continuación y responde a las siguientes preguntas:**
*   **¿Qué algoritmos de clasificación has probado?**
*   **¿Qué técnica/técnicas has utilizado para validar los algoritmos? En este caso práctico, ¿son validas todas las métricas que has utilizado?**
*   **¿Cuál de estos algoritmos funciona mejor? ¿Por qué?**



*Respuesta:*

- Se han probado los siguientes algoritmos: Regresión  Logística, Máquinas de Vectores de Soporte (SVM) - Lineales, Árboles de desición, clasificación ramdon forest, Árboles de Decisión - Gradient-boosted tree classifier y redes neuronales.
- En cuanto a las técnicas utilizadas para validar los algoritmos he empleado la validación del modelo, las clases reales empleadas comparadas con la predicción además de la matriz de confusión. En cuanto a los resultados esperados indicar que han sido satisfactorios y que los algoritmos utilizados no requerían ajustes adicionales.
- En este punto, indicar que todos algoritmos utilizados daban como resultado una precisión del 100%, valor dado en la evaluación del modelo (Accuracy), además de no presentar error en la matriz de confusión.